# SequoIA MPNet + SGD continuous deployment pipeline

The deployment section below continuously retrains a versioned SGD candidate, validates it on a fixed holdout, and moves the production pointer only when the candidate improves on the current production model.


## 1. Setup

In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

from hal_sequoia_extractor import SEQUOIA_AXES, SEQUOIA_SAXES

WORKSPACE = Path('/home/formation2026/Bureau/veille_test')
DATA_PATH = WORKSPACE / 'data/manual_labels_with_hal_metadata.csv'
ARTIFACT_DIR = WORKSPACE / 'mpnet_sgd_artifacts'
ARTIFACT_DIR.mkdir(exist_ok=True)
MODEL_NAME = 'sentence-transformers/all-mpnet-base-v2' 
RANDOM_STATE = 42
CANONICAL_LABELS = [
    'Core AI',
    'AI, Cybersecurity and Defense',
    'AI, Environment and Ocean',
    'No class',
]
PILLAR_MAP = {
    'Core AI': 'Core AI',
    'IA & cybersécurité': 'AI, Cybersecurity and Defense',
    'IA & environnement': 'AI, Environment and Ocean',
    'Non classifié': 'No class',
}
print(DATA_PATH)
print(ARTIFACT_DIR)

/home/formation2026/Bureau/veille_test/manual_labels_with_hal_metadata.csv
/home/formation2026/Bureau/veille_test/mpnet_sgd_artifacts


## 2. Load and prepare the curated training data

In [11]:
df = pd.read_csv(DATA_PATH, dtype={'halId_s': str})

def as_text(value):
    if pd.isna(value):
        return ''
    return str(value)

text_columns = ['title_s', 'abstract', 'keywords']
df['text_for_classification'] = df[text_columns].fillna('').astype(str).agg(' '.join, axis=1).str.replace(r'\s+', ' ', regex=True).str.strip()

training_df = df.copy()
print(f'Total records: {len(df):,}')
print(f'Training records: {len(training_df):,}')
print(training_df['manual_label'].value_counts())

Total records: 1,404
Training records: 1,404
manual_label
No class                         620
Core AI                          526
AI, environment and ocean        151
AI, cybersecurity and defense    107
Name: count, dtype: int64


In [15]:
df.head(1)


,halId_s,title_s,manual_label,abstract,keywords,text_for_classification
0,hal-04164424,Unravelling individual rhythmic abilities usin...,Core AI,Humans can easily extract the rhythm of a comp...,Human behaviour | Psychology,Unravelling individual rhythmic abilities usin...


## 3. Encode with MPNet and train SGDClassifier

In [18]:
encoder = SentenceTransformer(MODEL_NAME)
embeddings = encoder.encode(
    training_df['text_for_classification'].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True,
)

X_train, X_test, y_train, y_test = train_test_split(
    embeddings,
    training_df['manual_label'],
    test_size=0.25,
    stratify=training_df['manual_label'],
    random_state=RANDOM_STATE,
)

classifier = SGDClassifier(
    loss='log_loss',
    max_iter=2000,
    tol=1e-3,
    class_weight='balanced',
    random_state=RANDOM_STATE,
)
classifier.fit(X_train, y_train)
test_prediction = classifier.predict(X_test)
print(f'Accuracy: {accuracy_score(y_test, test_prediction):.3f}')
print(classification_report(y_test, test_prediction, zero_division=0))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/44 [00:00<?, ?it/s]

Accuracy: 0.775
                               precision    recall  f1-score   support

AI, cybersecurity and defense       0.60      0.67      0.63        27
    AI, environment and ocean       0.72      0.76      0.74        38
                      Core AI       0.81      0.76      0.79       131
                     No class       0.79      0.81      0.80       155

                     accuracy                           0.77       351
                    macro avg       0.73      0.75      0.74       351
                 weighted avg       0.78      0.77      0.78       351



## 4. Refit on all labels and save deployment artifacts

In [20]:
final_classifier = SGDClassifier(
    loss='log_loss',
    max_iter=2000,
    tol=1e-3,
    class_weight='balanced',
    random_state=RANDOM_STATE,
)
final_classifier.fit(embeddings, training_df['manual_label'].to_numpy())

joblib.dump(final_classifier, ARTIFACT_DIR / 'sgd_classifier.joblib')
encoder.save(str(ARTIFACT_DIR / 'mpnet_encoder'))
(ARTIFACT_DIR / 'metadata.json').write_text(
    json.dumps({
        'model_name': MODEL_NAME,
        'classifier': 'sklearn.linear_model.SGDClassifier',
        'loss': 'log_loss',
        'labels': CANONICAL_LABELS,
        'text_columns': text_columns,
        'subaxis_source': 'hal_sequoia_extractor.SEQUOIA_SAXES',
        'random_state': RANDOM_STATE,
    }, indent=2),
    encoding='utf-8',
)
print('Saved:', ARTIFACT_DIR / 'mpnet_encoder')
print('Saved:', ARTIFACT_DIR / 'sgd_classifier.joblib')
print('Saved:', ARTIFACT_DIR / 'metadata.json')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved: /home/formation2026/Bureau/veille_test/mpnet_sgd_artifacts/mpnet_encoder
Saved: /home/formation2026/Bureau/veille_test/mpnet_sgd_artifacts/sgd_classifier.joblib
Saved: /home/formation2026/Bureau/veille_test/mpnet_sgd_artifacts/metadata.json


## 5. Build and save pillar-conditioned sub-axis references

For each predicted pillar, only its `SEQUOIA_SAXES` entries are candidates. Both the article and
the sub-axis reference include the corresponding pillar description before MPNet cosine similarity.

In [23]:
subaxis_references = {}
for pillar, subaxes in SEQUOIA_SAXES.items():
    pillar_description = SEQUOIA_AXES[pillar]
    subaxis_names = list(subaxes)
    subaxis_texts = [
        f"{' '.join(keywords)} {pillar_description}"
        for keywords in subaxes.values()
    ]

    subaxis_references[pillar] = {
        'names': subaxis_names,
        'embeddings': encoder.encode(
            subaxis_texts,
            normalize_embeddings=True,
            show_progress_bar=False,
        ),
    }

joblib.dump(subaxis_references, ARTIFACT_DIR / 'subaxis_references.joblib')
print('Saved:', ARTIFACT_DIR / 'subaxis_references.joblib')
print({pillar: len(reference['names']) for pillar, reference in subaxis_references.items()})

Saved: /home/formation2026/Bureau/veille_test/mpnet_sgd_artifacts/subaxis_references.joblib
{'Core AI': 7, 'AI, Cybersecurity and Defense': 5, 'AI, Environment and Ocean': 3}


## 5. Server-ready prediction function

In [24]:
def predict_article(title='', abstract='', keywords='', journal=''):
    text = ' '.join(str(value or '') for value in [title, abstract, keywords, journal]).strip()
    article_embedding = encoder.encode([text], normalize_embeddings=True)

    pillar_probabilities = final_classifier.predict_proba(article_embedding)[0]
    pillar_index = int(pillar_probabilities.argmax())
    pillar = str(final_classifier.classes_[pillar_index])

    print(f'Predicted pillar: {pillar} (confidence: {pillar_probabilities[pillar_index]:.3f})')
    if pillar not in subaxis_references:
        print(f'Warning: No subaxis references found for pillar "{pillar}". Returning only pillar prediction.')
        return {
            'pillar': pillar,
            'pillar_confidence': float(pillar_probabilities[pillar_index]),
            'subaxis': 'No class',
            'subaxis_similarity': None,
            'pillar_probabilities': {str(label): float(score) for label, score in zip(final_classifier.classes_, pillar_probabilities)},
        }

    #pillar_description = SEQUOIA_AXES[pillar]
    #element_with_pillar = f'{text} {pillar_description}'
    element_without_pillar = text
    element_embedding = encoder.encode([element_without_pillar], normalize_embeddings=True)
    reference = subaxis_references[pillar]
    subaxis_scores = encoder.similarity(element_embedding, reference['embeddings']).cpu().numpy()[0]
    subaxis_index = int(subaxis_scores.argmax())
    subaxis_probabilities = np.exp(subaxis_scores) / np.sum(np.exp(subaxis_scores))  # Softmax to get probabilities
    return {
        'pillar': pillar,
        'pillar_confidence': float(pillar_probabilities[pillar_index]),
        'subaxis': reference['names'][subaxis_index],
        'subaxis_similarity': float(subaxis_scores[subaxis_index]),
        'pillar_probabilities': {str(label): float(score) for label, score in zip(final_classifier.classes_, pillar_probabilities)},
        'subaxis_probabilities': {str(label): float(score) for label, score in zip(reference['names'], subaxis_probabilities)}
        }

predict_article(
    title='AI-based intrusion detection for secure networks',
    abstract='We study anomaly detection and cyber defense using machine learning.',
    keywords='cybersecurity, intrusion detection, anomaly detection',
)

Predicted pillar: AI, cybersecurity and defense (confidence: 0.936)


{'pillar': 'AI, cybersecurity and defense',
 'pillar_confidence': 0.9359273910522461,
 'subaxis': 'No class',
 'subaxis_similarity': None,
 'pillar_probabilities': {'AI, cybersecurity and defense': 0.9359273910522461,
  'AI, environment and ocean': 0.014157047495245934,
  'Core AI': 0.011076272465288639,
  'No class': 0.03883932903409004}}